In [ ]:
from PIL import Image
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from mma.metrics.mma import get_mma
from mma.metrics.seg import get_seg
from mma.metrics.average_precision import get_ap_50
from mma.metrics.aji import get_aji
from mma.metrics.panoptic_quality import get_pq

In [ ]:
rand_colors = []
for i in range(4000):
    rand_colors.append(np.random.rand(3) * 255)

def convert_label_to_rainbow(label):
    label_rainbow = np.zeros((label.shape[0], label.shape[1], 3), dtype=np.uint8)
    for cell in np.unique(label):
        if cell == 0:
            continue #background
        label_rainbow[label == cell] = rand_colors[cell]

    return label_rainbow

In [ ]:
def get_metrics(gts, corrupted_gts, corruption_type=None):
    mmas = []
    mma_greedys = []
    segs = []
    ajis = []
    ap_50s = []
    pqs = []
    for i in tqdm(range(len(gts))):
        gt = gts[i]
        curr_currupted_gts = corrupted_gts[i]

        curr_mmas = []
        curr_mma_greedys = []
        curr_segs = []
        curr_ajis = []
        curr_ap_50s = []
        curr_pqs = []

        for j in range(len(curr_currupted_gts)):
            curr_mmas.append(round(get_mma(curr_currupted_gts[j], gt), 2))
            curr_mma_greedys.append(round(get_mma(curr_currupted_gts[j], gt, greedy=True), 2))
            curr_segs.append(round(get_seg(curr_currupted_gts[j], gt), 2))
            curr_ajis.append(round(get_aji(curr_currupted_gts[j], gt), 2))
            curr_ap_50s.append(round(get_ap_50(curr_currupted_gts[j], gt), 2))
            curr_pqs.append(round(get_pq(curr_currupted_gts[j], gt), 2))

        mmas.append(curr_mmas)
        mma_greedys.append(curr_mma_greedys)
        segs.append(curr_segs)
        ajis.append(curr_ajis)
        ap_50s.append(curr_ap_50s)
        pqs.append(curr_pqs)

    os.makedirs(f'../corruption_response_metrics/{corruption_type}', exist_ok=True)
    np.save(f'../corruption_response_metrics/{corruption_type}/mmas.npy', np.array(mmas))
    np.save(f'../corruption_response_metrics/{corruption_type}/mma_greedys.npy', np.array(mma_greedys))
    np.save(f'../corruption_response_metrics/{corruption_type}/segs.npy', np.array(segs))
    np.save(f'../corruption_response_metrics/{corruption_type}/ajis.npy', np.array(ajis))
    np.save(f'../corruption_response_metrics/{corruption_type}/ap_50s.npy', np.array(ap_50s))
    np.save(f'../corruption_response_metrics/{corruption_type}/pqs.npy', np.array(pqs))

    mean_mmas = np.mean(np.array(mmas), axis=0)
    mean_mma_greedys = np.mean(np.array(mma_greedys), axis=0)
    mean_segs = np.mean(np.array(segs), axis=0)
    mean_ajis = np.mean(np.array(ajis), axis=0)
    mean_ap_50s = np.mean(np.array(ap_50s), axis=0)
    mean_pqs = np.mean(np.array(pqs), axis=0)

    data = np.array([
        mean_mmas, mean_mma_greedys, mean_segs, mean_ajis, mean_ap_50s, mean_pqs
    ]).T

    df = pd.DataFrame(data, columns=['MMA', 'MMA Greedy', 'SEG', 'AJI', 'AP@50', 'PQ'])

    return df



In [ ]:
def plot_metrics(df, title):
    # Set global font to Arial
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.size'] = 16
    
    markers = ['o', 's', '^', 'D', 'v', 'x']

    ax = df.plot(
        marker='o',
        legend=False,
        color=[
            "#1f77b4",  # blue
            "#ff7f0e",  # orange
            "#2ca02c",  # green
            "#d62728",  # red
            "#9467bd",  # purple
            "#17becf"   # teal (replaces brown)
        ]
    )

    # 3. Apply one unique marker to each line
    for i, line in enumerate(ax.get_lines()):
        line.set_marker(markers[i])
        line.set_markersize(6) 

    ax.set_xlabel('Iteration', fontname='Arial')
    ax.set_ylabel('Score', fontname='Arial')
    ax.set_title(f'Metrics vs {title} Iteration', fontname='Arial')

    #Invert Legend Order and Make Opaque
    # handles, labels = ax.get_legend_handles_labels()
    # ax.legend(handles[::-1], labels[::-1], frameon=True, facecolor='white', framealpha=1)

    # Ensure tick labels are also Arial
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontname('Arial')
        
    plt.grid()
    plt.show()


In [ ]:
gts = np.load('../mma/datasets/LiveCell/test/anns.npy')

# Corruption Tests

## Mask Erosion

In [ ]:
from mma.corruption_functions.mask_erosion import erode_masks

In [ ]:
eroded_gts = []
for i in tqdm(range(gts.shape[0])):
    curr_eroded = erode_masks(gts[i], iter=9)
    eroded_gts.append(curr_eroded)

np.save('../corruption_responses/erosion.npy', np.array(eroded_gts))

for i in range(10):
    eroded_gt = eroded_gts[0][i]
    eroded_gt_rgb = convert_label_to_rainbow(eroded_gt)
    plt.imshow(eroded_gt_rgb)
    plt.show()

In [ ]:
eroded_gts = np.load('../corruption_responses/erosion.npy')
metric_df = get_metrics(gts, corrupted_gts=eroded_gts, corruption_type='erosion')
print(metric_df)
plot_metrics(metric_df, 'Erosion')

## Mask Dilation

In [ ]:
from mma.corruption_functions.mask_dilation import dilate_masks

In [ ]:
dilated_gts = []
for i in tqdm(range(gts.shape[0])):
    curr_dilated = dilate_masks(gts[i], iter=9)
    dilated_gts.append(curr_dilated)

np.save('../corruption_responses/dilation.npy', np.array(dilated_gts))

for i in range(10):
    dilated_gt = dilated_gts[0][i]
    dilated_gt_rgb = convert_label_to_rainbow(dilated_gt)
    plt.imshow(dilated_gt_rgb)
    plt.show()

In [ ]:
dilated_gts = np.load('../corruption_responses/dilation.npy')
metric_df = get_metrics(gts, corrupted_gts=dilated_gts, corruption_type='dilation')
print(metric_df)
plot_metrics(metric_df, 'Dilation')

## Fragmentation

In [ ]:
from mma.corruption_functions.mask_fragmentation import fragment_masks

In [ ]:
fragmented_gts = []
for i in tqdm(range(gts.shape[0])):
    curr_fragmented = fragment_masks(gts[i], iter=9)
    fragmented_gts.append(curr_fragmented)

np.save('../corruption_responses/fragmentation.npy', np.array(fragmented_gts))

for i in range(10):
    fragmented_gt = fragmented_gts[0][i]
    fragmented_gt_rgb = convert_label_to_rainbow(fragmented_gt)
    plt.imshow(fragmented_gt_rgb)
    plt.show()

In [ ]:
fragmented_gts = np.load('../corruption_responses/fragmentation.npy')
metric_df = get_metrics(gts, corrupted_gts=fragmented_gts, corruption_type='fragmentation')
print(metric_df)
plot_metrics(metric_df, 'Fragmentation')

## Clump Masks

In [ ]:
from mma.corruption_functions.mask_clumping import clump_masks

In [ ]:
clumped_gts = []
for i in tqdm(range(gts.shape[0])):
    curr_clumped = clump_masks(gts[i], iter=9)
    clumped_gts.append(curr_clumped)

np.save('../corruption_responses/clumping.npy', np.array(clumped_gts))

for i in range(10):
    clumped_gt = clumped_gts[0][i]
    clumped_gt_rgb = convert_label_to_rainbow(clumped_gt)
    plt.imshow(clumped_gt_rgb)
    plt.show()

In [ ]:
clumped_gts = np.load('../corruption_responses/clumping.npy')
metric_df = get_metrics(gts, corrupted_gts=clumped_gts, corruption_type='clumping')
print(metric_df)
plot_metrics(metric_df, 'Clumping')

## Random Removal

In [ ]:
from mma.corruption_functions.random_removal import remove_masks

In [ ]:
removed_gts = []
for i in tqdm(range(gts.shape[0])):
    curr_removed = remove_masks(gts[i], iter=9)
    removed_gts.append(curr_removed)

np.save('../corruption_responses/removal.npy', np.array(removed_gts))

for i in range(10):
    removed_gt = removed_gts[0][i]
    removed_gt_rgb = convert_label_to_rainbow(removed_gt)
    plt.imshow(removed_gt_rgb)
    plt.show()

In [ ]:
removed_gts = np.load('../corruption_responses/removal.npy')
metric_df = get_metrics(gts, corrupted_gts=removed_gts, corruption_type='removal')
print(metric_df)
plot_metrics(metric_df, 'Removal')

## Random Addition

In [ ]:
from mma.corruption_functions.random_addition import add_masks

In [ ]:
added_gts = []
for i in tqdm(range(gts.shape[0])):
    curr_added = add_masks(gts[i], iter=9)
    added_gts.append(curr_added)

np.save('../corruption_responses/addition.npy', np.array(added_gts))

for i in range(10):
    added_gt = added_gts[0][i]
    added_gt_rgb = convert_label_to_rainbow(added_gt)
    plt.imshow(added_gt_rgb)
    plt.show()

In [ ]:
added_gts = np.load('../corruption_responses/addition.npy')
metric_df = get_metrics(gts, corrupted_gts=added_gts, corruption_type='addition')
print(metric_df)
plot_metrics(metric_df, 'Addition')

In [ ]:
variable_size_added_gts = []
for i in tqdm(range(gts.shape[0])):
    curr_variable_size_added_gts = []
    for j in range(1, 11):
        curr_variable_size_added_gt = add_masks(gts[i], min_radius=j*4, max_radius=j*4, num_add_per_iter=10, iter=1)[1]
        curr_variable_size_added_gts.append(curr_variable_size_added_gt)
    variable_size_added_gts.append(curr_variable_size_added_gts)
        
np.save('../corruption_responses/variable_size_addition_new.npy', np.array(variable_size_added_gts))

for i in range(0, 10):
    added_gt = variable_size_added_gts[0][i]
    added_gt_rgb = convert_label_to_rainbow(added_gt)
    plt.imshow(added_gt_rgb)
    plt.show()

In [ ]:
variable_size_added_gts = np.load('../corruption_responses/variable_size_addition_new.npy')
metric_df = get_metrics(gts, corrupted_gts=variable_size_added_gts, corruption_type='variable_size_addition_new')
print(metric_df)
plot_metrics(metric_df, 'Addition Size')

## Mask Shifting

In [ ]:
from mma.corruption_functions.mask_shifting import shift_masks

In [ ]:
shifted_gts = []
for i in tqdm(range(gts.shape[0])):
    curr_shifted = shift_masks(gts[i], shift_mag_min=1, shift_mag_max=5, shift_prob=0.5, iter=9)
    shifted_gts.append(curr_shifted)

np.save('../corruption_responses/shifting.npy', np.array(shifted_gts))

for i in range(10):
    shifted_gt = shifted_gts[0][i]
    shifted_gt_rgb = convert_label_to_rainbow(shifted_gt)
    plt.imshow(shifted_gt_rgb)
    plt.show()

In [ ]:
shifted_gts = np.load('../corruption_responses/shifting.npy')
metric_df = get_metrics(gts, corrupted_gts=shifted_gts, corruption_type='shifting')
print(metric_df)
plot_metrics(metric_df, 'Shifting')

In [ ]:
variable_shifted_size_gts = []
for i in tqdm(range(gts.shape[0])):
    for j in range(1, 11):
        curr_variable_size_shifted_gt = shift_masks(gts[i], shift_mag_min=j, shift_mag_max=j+1, shift_prob=0.5, iter=1)[1]
        variable_shifted_size_gts.append(curr_variable_size_shifted_gt)
        
np.save('../corruption_responses/variable_size_shifting.npy', np.array(variable_shifted_size_gts))

for i in range(1, 11):
    shifted_gt = variable_shifted_size_gts[0][i]
    shifted_gt_rgb = convert_label_to_rainbow(shifted_gt)
    plt.imshow(shifted_gt_rgb)
    plt.show()

In [ ]:
variable_size_shifted_gts = np.load('../corruption_responses/variable_size_shifting.npy')
metric_df = get_metrics(gts, corrupted_gts=variable_size_shifted_gts, corruption_type='variable_size_shifting')
print(metric_df)
plot_metrics(metric_df, 'Shifting Size')

# Plot Creation

In [ ]:
corruption_type = 'addition' #dilation
mmas = np.load(f'../corruption_response_metrics/{corruption_type}/mmas.npy')
mma_greedys = np.load(f'../corruption_response_metrics/{corruption_type}/mma_greedys.npy')
segs = np.load(f'../corruption_response_metrics/{corruption_type}/segs.npy')
ajis = np.load(f'../corruption_response_metrics/{corruption_type}/ajis.npy')
ap_50s = np.load(f'../corruption_response_metrics/{corruption_type}/ap_50s.npy')
pqs = np.load(f'../corruption_response_metrics/{corruption_type}/pqs.npy')


if 'variable' in corruption_type:
    for i in reversed(range(mmas.shape[1])):
        if i == 0:
            mmas[:, i] = 1.0
            mma_greedys[:, i] = 1.0
            segs[:, i] = 1.0
            ajis[:, i] = 1.0
            ap_50s[:, i] = 1.0
            pqs[:, i] = 1.0
        else:
            mmas[:, i] = mmas[:, i-1]
            mma_greedys[:, i] = mma_greedys[:, i-1]
            segs[:, i] = segs[:, i-1]
            ajis[:, i] = ajis[:, i-1]
            ap_50s[:, i] = ap_50s[:, i-1]
            pqs[:, i] = pqs[:, i-1]


mean_mmas = np.mean(mmas, axis=0)
mean_mma_greedys = np.mean(mma_greedys, axis=0)
mean_segs = np.mean(segs, axis=0)
mean_ajis = np.mean(ajis, axis=0)
mean_ap_50s = np.mean(ap_50s, axis=0)
mean_pqs = np.mean(pqs, axis=0)

data = np.array([
    mean_ap_50s, mean_pqs, mean_segs, mean_ajis, mean_mma_greedys, mean_mmas 
]).T

df = pd.DataFrame(data, columns=['AP@50', 'PQ', 'SEG', 'AJI', 'MMA Greedy', 'MMA'])
print(df)
plot_metrics(df, corruption_type)